# Dashboard Olist — jawaban Q1–Q4 + freshness

Semua angka dibaca dari schema `mart.*` di PostgreSQL (bukan dari CSV mentah). Logika bisnis ada di `sql/mart/*.sql`; notebook ini hanya membaca dan menggambar. Query yang sama dipakai `outputs/marts_report.md`. Jalankan `make marts` dulu bila mart belum ada. Untuk melihat bulan tidak lengkap, ubah `COMPLETE_ONLY` menjadi `False`.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path('/home/jovyan')] if (p / 'olist').is_dir())
sys.path.insert(0, str(root))
%matplotlib inline
from olist import dashboard

COMPLETE_ONLY = True   # sembunyikan bulan tidak lengkap (2016-09/10/12, 2018-09/10)

In [ ]:
conn = dashboard.connect()
figs, insights = dashboard.build(conn, complete_only=COMPLETE_ONLY)
conn.close()
import matplotlib.pyplot as plt
plt.close('all')   # tampilkan tiap grafik sekali, di sel masing-masing
for k, v in insights.items():
    print(f'{k:6s} {v}\n')

## Q1 — Pendapatan per bulan dan kategori

In [ ]:
figs['q1_pendapatan']

## Q2 — Keterlambatan pengiriman dan skor review

In [ ]:
figs['q2_pengiriman']

## Q3 — Seller yang perlu ditindak

In [ ]:
figs['q3_seller']

## Q4 — Pelanggan (RFM)

In [ ]:
figs['q4_pelanggan']

## Freshness stream (T3)

In [ ]:
figs['freshness_stream']

## Penelusuran satu angka

Contoh: pendapatan Q1 = `sum(mart.sales_monthly.revenue)` = `sum(dw.fact_order_item.price)` untuk order bukan `canceled`/`unavailable`.

In [ ]:
import pandas as pd
conn = dashboard.connect()
pd.read_sql_query("""
SELECT (SELECT sum(revenue) FROM mart.sales_monthly) AS dari_mart,
       (SELECT sum(price) FROM dw.fact_order_item WHERE order_status NOT IN ('canceled','unavailable')) AS dari_fakta
""", conn)